<a href="https://colab.research.google.com/github/GabrielJ07/ConfiguratorAgent/blob/main/Copy_of_Circuittelligence_Core_Engine.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
"""
Circuittelligence Nervous System (CNS) - V0.1
PROPRIETARY AND CONFIDENTIAL

Implementation of the Strategic Stability Engine.
This module contains the strict Pydantic models and the minimal
loop required to demonstrate Knowledge Decay and Signal Promotion.
"""

from pydantic import BaseModel, Field
from typing import Literal, Set, Optional
from datetime import datetime, timezone
import uuid

# ==========================================
# STEP 2: FORMAL CONSTANTS
# ==========================================
DECAY_RATE = 0.05
REINFORCEMENT_GAIN = 0.08
CONTRADICTION_PENALTY = 0.10
JULES_MULTIPLIER = 1.5
PROMOTION_AUTHORITY_THRESHOLD = 0.65
PROMOTION_DIVERSITY_THRESHOLD = 3
PROMOTION_PERSISTENCE_THRESHOLD = 3
PIR_SEED_SCORE = 0.80

# ==========================================
# STEP 3: THREE CORE OBJECTS (PYDANTIC)
# ==========================================

class EvidenceEvent(BaseModel):
    """Immutable record of an inbound signal (reinforcement or contradiction)."""
    event_id: uuid.UUID = Field(default_factory=uuid.uuid4)
    claim_id: uuid.UUID
    context_id: uuid.UUID
    source_type: Literal["rss", "api", "manual", "internal", "jules_agent"]
    reinforcement: float = Field(ge=0.0, le=1.0, description="Degree of corroboration")
    contradiction: float = Field(ge=0.0, le=1.0, description="Degree of rebuttal")
    timestamp: datetime = Field(default_factory=lambda: datetime.now(timezone.utc))

class Claim(BaseModel):
    """The central unit of intelligence undergoing the vetting cycle."""
    id: uuid.UUID = Field(default_factory=uuid.uuid4)
    content: str
    authority_score: float = Field(default=0.1, ge=0.0, le=1.0)
    decay_survivals: int = Field(default=0)
    distinct_sources: Set[uuid.UUID] = Field(default_factory=set)
    is_promoted: bool = Field(default=False)
    is_pir: bool = Field(default=False)
    created_at: datetime = Field(default_factory=lambda: datetime.now(timezone.utc))
    updated_at: datetime = Field(default_factory=lambda: datetime.now(timezone.utc))

class PromotionDecision(BaseModel):
    """Audit record of a claim passing or failing the Promotion Board."""
    decision_id: uuid.UUID = Field(default_factory=uuid.uuid4)
    claim_id: uuid.UUID
    approved: bool
    reason: str
    timestamp: datetime = Field(default_factory=lambda: datetime.now(timezone.utc))


# ==========================================
# STEP 4: MINIMAL EXECUTION LOOP
# ==========================================

class CNSEngine:
    def __init__(self):
        self.audit_log = []

    def apply_decay(self, claim: Claim) -> Claim:
        """Applies temporal decay to a claim if it hasn't been reinforced."""
        if claim.is_promoted:
            return claim # Vetted goals do not decay organically in v0.1

        old_score = claim.authority_score
        claim.authority_score = max(0.0, claim.authority_score * (1 - DECAY_RATE))
        claim.decay_survivals += 1
        claim.updated_at = datetime.now(timezone.utc)

        print(f"[DECAY] Claim '{claim.content[:15]}...' decayed from {old_score:.3f} to {claim.authority_score:.3f}. Survivals: {claim.decay_survivals}")
        return claim

    def process_evidence(self, claim: Claim, evidence: EvidenceEvent) -> Claim:
        """Updates claim authority based on new evidence and Jules verification."""
        old_score = claim.authority_score

        # Determine asymmetric multiplier
        j_mult = JULES_MULTIPLIER if evidence.source_type == "jules_agent" else 1.0

        # Calculate impacts
        gain = evidence.reinforcement * REINFORCEMENT_GAIN
        penalty = evidence.contradiction * CONTRADICTION_PENALTY * j_mult

        # Apply equation: A_t+1 = clamp(A_t + gain - penalty, 0, 1)
        # Note: Decay is handled in a separate temporal cycle.
        new_score = old_score + gain - penalty
        claim.authority_score = max(0.0, min(1.0, new_score))

        # Track diversity
        claim.distinct_sources.add(evidence.context_id)
        claim.updated_at = datetime.now(timezone.utc)

        print(f"[EVIDENCE] Source: {evidence.source_type} | Gain: +{gain:.3f} | Penalty: -{penalty:.3f} | New Score: {claim.authority_score:.3f}")
        return claim

    def evaluate_promotion(self, claim: Claim) -> PromotionDecision:
        """The Promotion Board (Transition Trigger)."""
        if claim.is_promoted:
            return PromotionDecision(claim_id=claim.id, approved=True, reason="Already promoted.")

        gate_authority = claim.authority_score >= PROMOTION_AUTHORITY_THRESHOLD
        gate_diversity = len(claim.distinct_sources) >= PROMOTION_DIVERSITY_THRESHOLD
        gate_persistence = claim.decay_survivals >= PROMOTION_PERSISTENCE_THRESHOLD

        approved = gate_authority and gate_diversity and gate_persistence

        reason = (f"Auth: {claim.authority_score:.2f}/{PROMOTION_AUTHORITY_THRESHOLD} | "
                  f"Div: {len(claim.distinct_sources)}/{PROMOTION_DIVERSITY_THRESHOLD} | "
                  f"Pers: {claim.decay_survivals}/{PROMOTION_PERSISTENCE_THRESHOLD}")

        if approved:
            claim.is_promoted = True
            print(f"[PROMOTION BOARD] APPROVED: {reason}")
        else:
            print(f"[PROMOTION BOARD] DENIED: {reason}")

        decision = PromotionDecision(claim_id=claim.id, approved=approved, reason=reason)
        self.audit_log.append(decision)
        return decision

# ==========================================
# DEMONSTRATION: SYNTHETIC SIGNAL TEST
# ==========================================
if __name__ == "__main__":
    engine = CNSEngine()

    print("--- INITIATING CNS MINIMAL LOOP ---")
    # 1. Ingest an experimental claim
    target_claim = Claim(content="Competitor X is migrating to a Rust-based backend.")
    print(f"Created Claim: '{target_claim.content}' | Initial Score: {target_claim.authority_score}")

    # 2. Add corroborating evidence from 3 different sources to pass diversity gate
    for i in range(3):
        ev = EvidenceEvent(
            claim_id=target_claim.id,
            context_id=uuid.uuid4(),
            source_type="rss",
            reinforcement=0.9,
            contradiction=0.0
        )
        engine.process_evidence(target_claim, ev)

    # 3. Simulate time passing (Decay Cycles)
    for _ in range(3):
        engine.apply_decay(target_claim)

    # 4. First Promotion Attempt (Will likely fail on authority due to decay)
    engine.evaluate_promotion(target_claim)

    # 5. Heavy internal reinforcement
    ev_strong = EvidenceEvent(
        claim_id=target_claim.id,
        context_id=uuid.uuid4(),
        source_type="internal",
        reinforcement=1.0,
        contradiction=0.0
    )
    for _ in range(5): # Simulating persistent internal validation
        engine.process_evidence(target_claim, ev_strong)

    # 6. Jules Agent identifies a contradiction
    print("\n--- JULES AGENT INTERVENTION ---")
    jules_ev = EvidenceEvent(
        claim_id=target_claim.id,
        context_id=uuid.uuid4(),
        source_type="jules_agent",
        reinforcement=0.0,
        contradiction=0.8  # Jules found strong evidence this is false
    )
    engine.process_evidence(target_claim, jules_ev)

    # 7. Final Promotion Attempt
    print("\n--- FINAL EVALUATION ---")
    engine.evaluate_promotion(target_claim)